# PolicyLens AI: clause classifier upgrade

Run the cells **top to bottom** (Runtime -> Run all also works, but the first time go step by step).
**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.

What this notebook does:
1. Loads your clause data and the 300-row LLM-labelled (Claude) **test set** (never trained on).
2. Uses **Gemini** to label a few thousand extra clauses for training.
3. Checks how much Gemini agrees with the test-set labels (so we know the label noise).
4. Trains 3 kinds of models and compares them on the same test set.
5. Saves the best model for you to download.

Accuracy here means agreement with LLM-made labels, not human-verified ones. Say that in your report.

## Step 0: Install

In [ ]:
!pip -q install google-genai sentence-transformers

## Step 1: Upload the two data files
Click **Choose files** and select `clauses.csv` and `test_set_labeled.csv`.
(Or put both in the Colab file panel on the left and skip the upload.)

In [ ]:
import os
if not (os.path.exists("clauses.csv") and os.path.exists("test_set_labeled.csv")):
    from google.colab import files
    files.upload()
print(os.path.exists("clauses.csv"), os.path.exists("test_set_labeled.csv"))

Saving test_set_labeled.csv to test_set_labeled.csv
Saving clauses.csv to clauses.csv
True True


## Step 2: Settings and data preparation

In [ ]:
import pandas as pd, numpy as np, re, json, os, time
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ---- Settings you can change ----
GEMINI_MODEL = "auto"               # "auto" picks a model your key can use; or type a name yourself
CAP_PER_CLASS = 450                 # how many pool rows per keyword-guess class go to Gemini (total ~3,600)
BATCH_SIZE = 20                     # clauses per Gemini request
SECONDS_BETWEEN_CALLS = 7           # free key allows ~10 requests/minute; raise this if you hit 429 errors
USE_ONLY_HIGH_CONFIDENCE = True     # train only on Gemini labels it was confident about
SEED = 42
# ---------------------------------

LABELS = ["claims","coverage","definition","exclusion","limit","other","premium","renewal","waiting_period"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

norm = lambda s: re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

raw = pd.read_csv("clauses.csv", encoding="utf-8-sig")
raw["row_id"] = raw.index
raw["norm"] = raw.text.map(norm)

test = pd.read_csv("test_set_labeled.csv").rename(columns={"claude_label": "label"})
test_norms = set(test.text.map(norm))

is_rev = raw.reviewed.astype(str).str.lower().eq("yes")
human = raw[is_rev & ~raw.norm.isin(test_norms)].drop_duplicates("norm").copy()

pool = raw[~is_rev & (raw.text.str.len() >= 40)]
pool = pool[~pool.row_id.isin(test.row_id) & ~pool.norm.isin(test_norms)].drop_duplicates("norm")

parts = []
for k, d in pool.groupby("suggested_label"):
    parts.append(d.sample(min(len(d), CAP_PER_CLASS), random_state=SEED))
sample = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("human-reviewed rows :", len(human))
print("pool rows available :", len(pool))
print("pool rows to label  :", len(sample))
print("test rows           :", len(test))
print(test.label.value_counts().to_string())

human-reviewed rows : 148
pool rows available : 9846
pool rows to label  : 3755
test rows           : 300
label
definition        69
other             49
exclusion         42
claims            33
coverage          29
limit             24
renewal           22
waiting_period    19
premium           13


## Step 3: Gemini setup
1. Go to https://aistudio.google.com/apikey and click **Create API key** (free).
2. In Colab, click the **key icon** in the left sidebar -> **Add new secret** -> name it `GEMINI_API_KEY`, paste the key, and switch on **Notebook access**.

Note: the Gemini Plus/Pro subscription is for the Gemini app. API use is separate and is covered by this free key.

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

def call_gemini(prompt):
    r = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config={"temperature": 0, "response_mime_type": "application/json"},
    )
    return r.text

# Try models one by one with a tiny test request and use the first one that really works on your free key.
PREFERRED = ["gemini-3.1-flash-lite", "gemini-3.5-flash-lite", "gemini-flash-lite-latest", "gemini-2.5-flash-lite",
             "gemini-3.5-flash", "gemini-flash-latest", "gemini-2.5-flash", "gemini-3.6-flash"]
candidates = PREFERRED if GEMINI_MODEL in (None, "", "auto") else [GEMINI_MODEL]
GEMINI_MODEL = None
for name in candidates:
    GEMINI_MODEL = name
    try:
        out = call_gemini('Reply with exactly this JSON: {"ok": true}')
        print("WORKS:", name, "->", out.strip()[:40])
        break
    except Exception as e:
        print("not usable:", name, "->", str(e)[:100].replace("\n", " "))
        GEMINI_MODEL = None
        time.sleep(2)
if GEMINI_MODEL is None:
    raise RuntimeError("None of the models worked. Paste the messages above to Claude.")
print("Using model:", GEMINI_MODEL)

WORKS: gemini-3.1-flash-lite -> {"ok": true}
Using model: gemini-3.1-flash-lite


In [ ]:
GUIDE = """You label paragraphs from Indian health-insurance policy documents.
Give EACH paragraph exactly one label, chosen by what the paragraph is mainly about:

claims         - how to make/notify/settle a claim, required documents, cashless process, pre-authorization, claim timelines, fraud-in-a-claim clauses
coverage       - what the insurer WILL pay for: benefits, covers, what is included, how a benefit works
definition     - defines a term ("X means ..."), including medical/critical-illness definitions and their diagnostic criteria, hospital definitions
exclusion      - what is NOT covered: exclusion lists, excluded treatments/conditions, "shall not be liable", items not payable
limit          - caps and money limits: sum insured, sub-limits, deductible, co-payment, room-rent limits, benefit tables, proportionate deduction
other          - headings, page headers/footers, insurer address/contact/ombudsman/grievance text, general legal conditions (notices, disclosure, arbitration), fragments with no clear topic
premium        - premium amount, payment mode, instalments, premium refunds, premium discounts/loading
renewal        - renewal, grace period, portability/migration, cancellation of policy, free-look, withdrawal of product, continuity of cover
waiting_period - waiting periods, pre-existing disease waiting, cooling-off, specific-disease waiting lists, time-bound exclusions, reduction of waiting period

Rules: if a paragraph mixes topics, pick the dominant one. Broken fragments with no clear topic -> other.
Give confidence "high" if the paragraph clearly fits one label, otherwise "low".
Reply with ONLY a JSON list like [{"id": 0, "label": "claims", "confidence": "high"}, ...] covering every id."""

def make_prompt(rows):
    items = [{"id": int(i), "text": str(t)[:1200]} for i, t in rows]
    return GUIDE + "\n\nParagraphs:\n" + json.dumps(items, ensure_ascii=False)

def parse(txt):
    txt = txt.strip()
    txt = re.sub(r"^```(?:json)?|```$", "", txt, flags=re.M).strip()
    out = {}
    for o in json.loads(txt):
        lab = str(o.get("label", "")).strip().lower()
        if lab in label2id:
            out[int(o["id"])] = (lab, str(o.get("confidence", "low")).lower())
    return out

def label_rows(rows, tries=5):
    """rows: list of (id, text). Returns {id: (label, confidence)}. Retries on errors / rate limits."""
    for attempt in range(tries):
        try:
            res = parse(call_gemini(make_prompt(rows)))
            return res
        except Exception as e:
            if any(k in str(e) for k in ("404", "NOT_FOUND", "API_KEY", "PERMISSION_DENIED", "403", "limit: 0")):
                raise RuntimeError(f"Gemini setup problem, not worth retrying: {str(e)[:300]}")
            wait = 20 * (attempt + 1)
            print(f"  problem ({str(e)[:80]}) - waiting {wait}s and retrying")
            time.sleep(wait)
    return {}

## Step 4: Check Gemini against the test-set labels (LLM-made) (about 15 requests)
This tells us how noisy Gemini's labels are. If it agrees with my labels ~70%+ of the time, using it for training is reasonable.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

rows = list(zip(test.tid, test.text))
pred = {}
for i in range(0, len(rows), BATCH_SIZE):
    pred.update(label_rows(rows[i:i+BATCH_SIZE]))
    time.sleep(SECONDS_BETWEEN_CALLS)

test["gemini_label"] = test.tid.map(lambda t: pred.get(int(t), ("other","low"))[0])
test["gemini_conf"]  = test.tid.map(lambda t: pred.get(int(t), ("other","low"))[1])
print("Gemini vs Claude-labelled test set -> accuracy:", round(accuracy_score(test.label, test.gemini_label), 3),
      " macro-F1:", round(f1_score(test.label, test.gemini_label, average="macro"), 3))
hi = test[test.gemini_conf == "high"]
print(f"On the {len(hi)} 'high confidence' rows -> accuracy:", round(accuracy_score(hi.label, hi.gemini_label), 3))
print(classification_report(test.label, test.gemini_label, zero_division=0))

  problem (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp) - waiting 20s and retrying
  problem (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp) - waiting 20s and retrying
  problem (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp) - waiting 20s and retrying
Gemini vs Claude-labelled test set -> accuracy: 0.86  macro-F1: 0.848
On the 292 'high confidence' rows -> accuracy: 0.86
                precision    recall  f1-score   support

        claims       0.76      0.97      0.85        33
      coverage       0.75      0.72      0.74        29
    definition       0.96      0.93      0.94        69
     exclusion       1.00      0.79      0.88        42
         limit       0.84      0.88      0.86        24
         other       0.93      0.78      0.84        49
       premium       0.92      0.85      0.88        13
       renewal       0.71      0.91      0.80        22
waiting_perio

## Step 5: Label the training pool with Gemini
This takes a while (about 180 requests, roughly 20 to 30 minutes on the free key). It **saves progress** to `pool_labeled.csv`, so if it stops (rate limit, disconnect), just run this cell again and it continues.

In [ ]:
OUT = Path("pool_labeled.csv")
done = pd.read_csv(OUT) if OUT.exists() else pd.DataFrame(columns=["row_id","gemini_label","gemini_conf"])
have = set(done.row_id.astype(int))
todo = sample[~sample.row_id.isin(have)]
print("already labelled:", len(have), "| still to do:", len(todo))

recs = done.to_dict("records")
rows = list(zip(todo.row_id, todo.text))
for i in range(0, len(rows), BATCH_SIZE):
    chunk = rows[i:i+BATCH_SIZE]
    res = label_rows(chunk)
    for rid, (lab, conf) in res.items():
        recs.append({"row_id": rid, "gemini_label": lab, "gemini_conf": conf})
    pd.DataFrame(recs).to_csv(OUT, index=False)
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"  {len(recs)} labelled so far")
    time.sleep(SECONDS_BETWEEN_CALLS)

pool_lab = pd.read_csv(OUT)
print("DONE. labelled:", len(pool_lab))
print(pool_lab.gemini_label.value_counts().to_string())
print(pool_lab.gemini_conf.value_counts().to_string())

already labelled: 0 | still to do: 3755
  problem (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp) - waiting 20s and retrying
  20 labelled so far
  220 labelled so far
  problem (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp) - waiting 20s and retrying
  420 labelled so far
  620 labelled so far
  820 labelled so far
  1020 labelled so far
  1219 labelled so far
  1419 labelled so far
  1619 labelled so far
  problem (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp) - waiting 20s and retrying
  1819 labelled so far
  2019 labelled so far
  2218 labelled so far
  2417 labelled so far
  2617 labelled so far
  2817 labelled so far
  3017 labelled so far
  3217 labelled so far
  3417 labelled so far
  3617 labelled so far
DONE. labelled: 3752
gemini_label
definition        884
claims            465
other             451
exclusion         417
renewal           387
coverage          382
lim

## Step 6: Build the training set

In [ ]:
pl = pd.read_csv("pool_labeled.csv").merge(raw[["row_id","text","policy_id","norm"]], on="row_id")
pl = pl.rename(columns={"gemini_label": "label"})
if USE_ONLY_HIGH_CONFIDENCE:
    pl = pl[pl.gemini_conf == "high"]

train_all = pd.concat([
    human[["text","label","policy_id"]].assign(src="human"),
    pl[["text","label","policy_id"]].assign(src="gemini"),
]).reset_index(drop=True)
train_all["label"] = train_all.label.str.strip().str.lower()
train_all = train_all[train_all.label.isin(LABELS)]

# validation split by document (used only to pick the best training epoch)
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.12, random_state=SEED)
tr_idx, va_idx = next(gss.split(train_all, groups=train_all.policy_id))
train_df, val_df = train_all.iloc[tr_idx].reset_index(drop=True), train_all.iloc[va_idx].reset_index(drop=True)
print("train:", len(train_df), "| validation:", len(val_df), "| test:", len(test))
print(train_df.label.value_counts().to_string())

train: 3396 | validation: 363 | test: 300
label
definition        761
claims            439
exclusion         407
coverage          370
renewal           354
other             338
limit             279
waiting_period    233
premium           215


## Step 7: Models
Every model is scored on the same 300-row test set. The test set is only used for the final scores, never for training or picking epochs.

In [ ]:
results = {}
def report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred); f1 = f1_score(y_true, y_pred, average="macro")
    results[name] = {"accuracy": round(acc,4), "macro_f1": round(f1,4)}
    print(f"{name}: accuracy={acc:.3f}  macro-F1={f1:.3f}")

# 7a. Simple baseline: TF-IDF + Logistic Regression on the big training set
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
tfidf = Pipeline([("t", TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, min_df=2)),
                  ("c", LogisticRegression(max_iter=3000, class_weight="balanced", C=10))])
tfidf.fit(pd.concat([train_df, val_df]).text, pd.concat([train_df, val_df]).label)
report("tfidf_logreg", test.label, tfidf.predict(test.text))

tfidf_logreg: accuracy=0.770  macro-F1=0.745


In [ ]:
# 7b. Sentence embeddings (frozen) + Logistic Regression
from sentence_transformers import SentenceTransformer
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
enc = SentenceTransformer("BAAI/bge-base-en-v1.5", device=dev)
enc.max_seq_length = 256
def emb(texts): return enc.encode(list(texts), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
full = pd.concat([train_df, val_df])
Xtr, Xte = emb(full.text), emb(test.text)
emb_lr = LogisticRegression(max_iter=3000, class_weight="balanced", C=5).fit(Xtr, full.label)
report("bge_embeddings_logreg", test.label, emb_lr.predict(Xte))

device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

bge_embeddings_logreg: accuracy=0.790  macro-F1=0.778


In [ ]:
# 7c. Fine-tune transformers (this is where the GPU matters)
import copy, torch, torch.nn as nn
dev = "cuda" if torch.cuda.is_available() else "cpu"
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

def finetune(model_name, epochs=6, lr=3e-5, bs=16, max_len=256):
    torch.manual_seed(SEED)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(LABELS), id2label=id2label, label2id=label2id).to(dev)
    ytr = train_df.label.map(label2id).values
    cnt = np.bincount(ytr, minlength=len(LABELS)).astype(float)
    w = torch.tensor(len(ytr) / (len(LABELS) * np.maximum(cnt, 1)), dtype=torch.float, device=dev)
    lossf = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(train_df) / bs))
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(dev == "cuda"))

    def predict(texts):
        model.eval(); out = []
        texts = list(texts)
        with torch.no_grad():
            for i in range(0, len(texts), 64):
                b = tok(texts[i:i+64], truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(dev)
                with torch.autocast(device_type="cuda", enabled=(dev == "cuda")):
                    out.append(model(**b).logits.float().argmax(-1).cpu().numpy())
        return np.concatenate(out)

    best_f1, best_state = -1, None
    for ep in range(epochs):
        model.train()
        perm = np.random.RandomState(SEED + ep).permutation(len(train_df))
        for i in range(0, len(perm), bs):
            idx = perm[i:i+bs]
            b = tok(list(train_df.text.values[idx]), truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(dev)
            y = torch.tensor(ytr[idx], device=dev)
            with torch.autocast(device_type="cuda", enabled=(dev == "cuda")):
                logits = model(**b).logits
            loss = lossf(logits.float(), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        vf1 = f1_score(val_df.label.map(label2id), predict(val_df.text), average="macro")
        print(f"  epoch {ep+1}/{epochs}  validation macro-F1 = {vf1:.3f}")
        if vf1 > best_f1:
            best_f1, best_state = vf1, copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
    model.load_state_dict(best_state)
    pred = predict(test.text)
    return model, tok, [id2label[i] for i in pred]

trained = {}
for name in ["nlpaueb/legal-bert-base-uncased", "BAAI/bge-base-en-v1.5"]:
    print("Training", name)
    m, t, p = finetune(name)
    report("finetune_" + name.split("/")[-1], test.label, p)
    trained[name] = (m, t)

Training nlpaueb/legal-bert-base-uncased


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

  epoch 1/6  validation macro-F1 = 0.769
  epoch 2/6  validation macro-F1 = 0.806
  epoch 3/6  validation macro-F1 = 0.796
  epoch 4/6  validation macro-F1 = 0.820
  epoch 5/6  validation macro-F1 = 0.807
  epoch 6/6  validation macro-F1 = 0.835
finetune_legal-bert-base-uncased: accuracy=0.827  macro-F1=0.796
Training BAAI/bge-base-en-v1.5


[transformers] You passed `num_labels=9` which is incompatible to the `id2label` map of length `1`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: BAAI/bge-base-en-v1.5
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1/6  validation macro-F1 = 0.797
  epoch 2/6  validation macro-F1 = 0.805
  epoch 3/6  validation macro-F1 = 0.784
  epoch 4/6  validation macro-F1 = 0.807
  epoch 5/6  validation macro-F1 = 0.801
  epoch 6/6  validation macro-F1 = 0.796
finetune_bge-base-en-v1.5: accuracy=0.823  macro-F1=0.805


**Known evaluation caveats (added after review):** the test set's `row_id`s fall in per-document blocks, and most Gemini-labelled pool rows come from the same policies, so train and test share documents. Cell 22 also picks the final model by *test* macro-F1. For a cleaner rerun: split by `policy_id` so no document is in both train and test, choose the model on validation F1 only, and report the test score once.


## Step 8: Compare, inspect, save the best

In [ ]:
res = pd.DataFrame(results).T.sort_values("macro_f1", ascending=False)
res.loc["(old) DistilBERT from repo"] = [0.2895, 0.3048]  # old numbers, measured on a different, tiny test set
res.loc["(old) keyword rules"] = [0.4633, np.nan]
display(res)

,accuracy,macro_f1
finetune_bge-base-en-v1.5,0.8233,0.8046
finetune_legal-bert-base-uncased,0.8267,0.7965
bge_embeddings_logreg,0.7900,0.7783
tfidf_logreg,0.7700,0.7453
(old) DistilBERT from repo,0.2895,0.3048
(old) keyword rules,0.4633,NaN


In [ ]:
from sklearn.metrics import confusion_matrix
ft = {k: v for k, v in results.items() if k.startswith("finetune_")}
best_key = max(ft, key=lambda k: ft[k]["macro_f1"])
best_name = [n for n in trained if n.split("/")[-1] == best_key.replace("finetune_", "")][0]
model, tok = trained[best_name]
print("Best fine-tuned model:", best_name)

model.eval(); preds = []
with torch.no_grad():
    for i in range(0, len(test), 64):
        b = tok(list(test.text.values[i:i+64]), truncation=True, max_length=256, padding=True, return_tensors="pt").to(dev)
        preds.append(model(**b).logits.argmax(-1).cpu().numpy())
preds = [id2label[i] for i in np.concatenate(preds)]
print(classification_report(test.label, preds, zero_division=0))
print(pd.DataFrame(confusion_matrix(test.label, preds, labels=LABELS), index=LABELS, columns=LABELS))

out_dir = "clause-classifier"
model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
json.dump(label2id, open(f"{out_dir}/labels.json", "w"), indent=2)
res.to_csv("results_summary.csv")
!zip -qr clause-classifier.zip clause-classifier results_summary.csv pool_labeled.csv
from google.colab import files
files.download("clause-classifier.zip")

Best fine-tuned model: BAAI/bge-base-en-v1.5
                precision    recall  f1-score   support

        claims       0.70      1.00      0.82        33
      coverage       0.64      0.62      0.63        29
    definition       0.94      0.97      0.96        69
     exclusion       0.94      0.79      0.86        42
         limit       0.87      0.83      0.85        24
         other       0.91      0.59      0.72        49
       premium       0.75      0.92      0.83        13
       renewal       0.72      0.82      0.77        22
waiting_period       0.74      0.89      0.81        19

      accuracy                           0.82       300
     macro avg       0.80      0.83      0.80       300
  weighted avg       0.84      0.82      0.82       300

                claims  coverage  definition  exclusion  limit  other  \
claims              33         0           0          0      0      0   
coverage             4        18           0          0      2      0   
defin

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>